# Full Pipeline: HL7 v2 → FHIR R4

Raw ADT^A01 messages → parsed segments → crossmapped FHIR resources →
transaction Bundle + NDJSON.

> **Prerequisites** — `pip install "portiere-health[polars]"`. This notebook is
> **fully offline**: no model downloads, no network calls, no API keys. It uses
> the bundled ICD-10-CM / LOINC / RxNorm vocabulary subset and BM25 lexical
> retrieval, and asserts the no-egress guarantee with `offline=True`.
> Every cell is deterministic — run top to bottom.

In [1]:
import json
import tempfile
from pathlib import Path

import polars as pl

import portiere
from portiere import __version__

WORK = Path(tempfile.mkdtemp(prefix="portiere_nb_"))
print(f"portiere v{__version__} — working dir: {WORK}")

portiere v0.4.0 — working dir: /var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_xiskm4zz


## Step 1 — Raw HL7 v2 messages

An ADT^A01 (admission) feed as it arrives from an interface engine — ER7
pipe-delimited text. We parse the segments Portiere's `hl7v2_2.5.1` standard
models (PID, PV1, DG1, OBX).

In [2]:
ADT_A01 = """MSH|^~\\&|HIS|GH|EHR|GH|20240115103000||ADT^A01|MSG0001|P|2.5.1
PID|1||MRN-10001||DOE^JANE||19850322|F|||12 Orchard Rd^^Klang^SGR
PV1|1|I||EMER|||||||||||||||VN-5001|||||||||||||||||||||||||20240115102500
DG1|1||I10|Essential (primary) hypertension|20240115|A
OBX|1|NM|2160-0^Creatinine||88|umol/L|||||F|||20240115103500"""

ADT_A01_B = """MSH|^~\\&|HIS|GH|EHR|GH|20240116090000||ADT^A01|MSG0002|P|2.5.1
PID|1||MRN-10002||TAN^AH KOW||19701130|M|||8 Jalan Besar^^Shah Alam^SGR
PV1|1|O||ELEC|||||||||||||||VN-5002|||||||||||||||||||||||||20240116085500
DG1|1||E11.9|Type 2 diabetes mellitus without complications|20240116|A
OBX|1|NM|4548-4^HbA1c||7.9|%|||||F|||20240116091500"""
print(ADT_A01.splitlines()[1])

PID|1||MRN-10001||DOE^JANE||19850322|F|||12 Orchard Rd^^Klang^SGR


In [3]:
def parse_hl7v2(message: str) -> dict[str, list[dict]]:
    """Parse pipe-delimited HL7 v2 (ER7) into segment dicts keyed for the
    Portiere hl7v2_2.5.1 standard (PID/PV1/DG1/OBX field names)."""
    FIELDS = {
        # segment -> {field_index: portiere_field_name}
        "PID": {3: "patient_id", 5: "patient_name", 7: "date_of_birth",
                8: "administrative_sex", 11: "patient_address"},
        "PV1": {2: "patient_class", 4: "admission_type", 19: "visit_number",
                44: "admit_date_time"},
        "DG1": {3: "diagnosis_code", 4: "diagnosis_description",
                5: "diagnosis_date_time", 6: "diagnosis_type"},
        "OBX": {2: "value_type", 3: "observation_identifier",
                5: "observation_value", 6: "units",
                14: "date_time_of_observation"},
    }
    out: dict[str, list[dict]] = {}
    for line in message.strip().splitlines():
        parts = line.strip().split("|")
        seg = parts[0]
        if seg not in FIELDS:
            continue
        record = {}
        for idx, name in FIELDS[seg].items():
            if idx < len(parts) and parts[idx]:
                record[name] = parts[idx]
        out.setdefault(seg, []).append(record)
    return out

In [4]:
messages = [parse_hl7v2(ADT_A01), parse_hl7v2(ADT_A01_B)]
print(json.dumps(messages[0], indent=2))

{
  "PID": [
    {
      "patient_id": "MRN-10001",
      "patient_name": "DOE^JANE",
      "date_of_birth": "19850322",
      "administrative_sex": "F",
      "patient_address": "12 Orchard Rd^^Klang^SGR"
    }
  ],
  "PV1": [
    {
      "patient_class": "I",
      "admission_type": "EMER",
      "visit_number": "VN-5001",
      "admit_date_time": "20240115102500"
    }
  ],
  "DG1": [
    {
      "diagnosis_code": "I10",
      "diagnosis_description": "Essential (primary) hypertension",
      "diagnosis_date_time": "20240115",
      "diagnosis_type": "A"
    }
  ],
  "OBX": [
    {
      "value_type": "NM",
      "observation_identifier": "2160-0^Creatinine",
      "observation_value": "88",
      "units": "umol/L",
      "date_time_of_observation": "20240115103500"
    }
  ]
}


## Step 2 — Map segments to FHIR resources

`CrossStandardMapper("hl7v2_2.5.1", "fhir_r4")` applies the shipped crossmap
(based on the HL7 v2-to-FHIR implementation guide): PID→Patient, PV1→Encounter,
DG1→Condition, OBX→Observation — with per-field transforms (`v2_gender`,
`fhir_date`, `codeable_concept`).

In [5]:
from portiere.local.cross_mapper import CrossStandardMapper, list_crossmaps

print("shipped crossmaps:")
for cm in list_crossmaps():
    print(f"  {cm['source']} -> {cm['target']}")

mapper = CrossStandardMapper("hl7v2_2.5.1", "fhir_r4")
print()
print("entity map:", mapper.get_entity_map())

shipped crossmaps:
  fhir_r4 -> omop_cdm_v5.4
  fhir_r4 -> openehr_1.0.4
  hl7v2_2.5.1 -> fhir_r4
  omop_cdm_v5.4 -> fhir_r4
  omop_cdm_v5.4 -> openehr_1.0.4



entity map: {'PID': 'Patient', 'PV1': 'Encounter', 'DG1': 'Condition', 'OBX': 'Observation', 'OBR': 'DiagnosticReport', 'RXE': 'MedicationRequest', 'AL1': 'AllergyIntolerance'}


In [6]:
entity_map = mapper.get_entity_map()

resources = []
for msg in messages:
    for seg_name, records in msg.items():
        for rec in records:
            mapped = mapper.map_record(seg_name, rec)
            # map_record returns the field dict; the resource type comes from
            # the crossmap's entity map. Drop empty values for clean resources.
            resources.append({
                "resourceType": entity_map[seg_name],
                **{k: v for k, v in mapped.items() if v not in (None, "", {})},
            })

for r in resources:
    print(f"  {r.get('resourceType', '?'):14s} {json.dumps(r, default=str)[:90]}")

  Patient        {"resourceType": "Patient", "name": "DOE^JANE", "birthDate": "19850322", "gender": "female
  Encounter      {"resourceType": "Encounter", "identifier": "VN-5001", "class": "IMP", "type": {"coding": 
  Condition      {"resourceType": "Condition", "code": "Essential (primary) hypertension", "onsetDateTime":
  Observation    {"resourceType": "Observation", "code": {"coding": [{"system": "http://loinc.org", "code":
  Patient        {"resourceType": "Patient", "name": "TAN^AH KOW", "birthDate": "19701130", "gender": "male
  Encounter      {"resourceType": "Encounter", "identifier": "VN-5002", "class": "AMB", "type": {"coding": 
  Condition      {"resourceType": "Condition", "code": "Type 2 diabetes mellitus without complications", "o
  Observation    {"resourceType": "Observation", "code": {"coding": [{"system": "http://loinc.org", "code":


## Step 3 — Bundle + NDJSON export

Ready for a FHIR server (`POST /` transaction) or bulk import. Profile
validation (`--fhir-profile us-core-6.1.0`) plugs in before writing when you
need conformance guarantees.

In [7]:
from portiere.export.fhir.bundle import to_transaction_bundle
from portiere.export.fhir.ndjson import to_ndjson_files

bundle = to_transaction_bundle(resources)
print(f"Bundle: {len(bundle['entry'])} entries")
print(json.dumps(bundle["entry"][0], indent=2, default=str)[:500])

nd = to_ndjson_files(resources, out_dir=WORK / "ndjson")
print("NDJSON files:", [p.name for p in nd])

Bundle: 8 entries
{
  "fullUrl": "urn:uuid:a6ee1d95-f530-4df7-999e-55a0d6812b45",
  "resource": {
    "resourceType": "Patient",
    "name": "DOE^JANE",
    "birthDate": "19850322",
    "gender": "female",
    "address": "12 Orchard Rd^^Klang^SGR"
  },
  "request": {
    "method": "POST",
    "url": "Patient"
  }
}
NDJSON files: ['Patient.ndjson', 'Encounter.ndjson', 'Condition.ndjson', 'Observation.ndjson']


## Step 4 — The mapping report

`get_mapping_report()` documents exactly which fields the crossmap covers —
paste it into your interface spec. Unmapped v2 fields are visible, not
silently dropped.

In [8]:
report = mapper.get_mapping_report()
print(json.dumps(report, indent=2, default=str)[:900])

{
  "source_standard": "hl7v2_2.5.1",
  "target_standard": "fhir_r4",
  "entity_mappings": [
    {
      "source": "PID",
      "target": "Patient"
    },
    {
      "source": "PV1",
      "target": "Encounter"
    },
    {
      "source": "DG1",
      "target": "Condition"
    },
    {
      "source": "OBX",
      "target": "Observation"
    },
    {
      "source": "OBR",
      "target": "DiagnosticReport"
    },
    {
      "source": "RXE",
      "target": "MedicationRequest"
    },
    {
      "source": "AL1",
      "target": "AllergyIntolerance"
    }
  ],
  "field_mappings": 45,
  "unmapped_source_fields": [
    "PID.phone_business",
    "PID.primary_language",
    "PID.birth_place",
    "PV1.assigned_location",
    "PV1.referring_doctor",
    "PV1.discharge_date_time",
    "DG1.diagnosis_coding_method",
    "DG1.diagnosis_priority",
    "OBX.value_type",
    "OBX.observation_sub_
